In [2]:
import os
os.environ['USE_TF']='0'
os.environ['USE_FLAX']='0'
GPU = 1
import os, gc, re, ast, io, contextlib, warnings
from pathlib import Path
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar
warnings.filterwarnings('ignore', category=UserWarning)
set_seed(42)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
ROOT = Path('/tf/wjy')
MODEL_PATH = ROOT / 'models/Meta-Llama-3-8B-Instruct'
assert torch.cuda.is_available(), 'CUDA is required'
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True, padding_side='left')
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, local_files_only=True,
    torch_dtype=torch.float16, device_map={'': GPU}, attn_implementation='sdpa')
model.eval()
assert not getattr(model, 'is_quantized', False)
assert {p.dtype for p in model.parameters() if p.is_floating_point()} == {torch.float16}
EOS_IDS = list({tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids('<|eot_id|>')})
@torch.inference_mode()
def generate_batch(messages, max_new_tokens):
    prompts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages]
    inputs = tokenizer(prompts, return_tensors='pt', padding=True, add_special_tokens=False)
    assert inputs.input_ids.shape[1] + max_new_tokens <= model.config.max_position_embeddings
    inputs = {k: v.to(model.device) for k,v in inputs.items()}
    output = model.generate(**inputs, do_sample=False, temperature=None, top_p=None,
        max_new_tokens=max_new_tokens, eos_token_id=EOS_IDS, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.batch_decode(output[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
def release_model():
    global model
    del model
    gc.collect()
    torch.cuda.empty_cache()


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 112.00 MiB. GPU 1 has a total capacity of 47.40 GiB of which 61.06 MiB is free. Process 28665 has 40.52 GiB memory in use. Including non-PyTorch memory, this process has 6.81 GiB memory in use. Of the allocated memory 6.37 GiB is allocated by PyTorch, and 145.79 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [2]:
import json
from pathlib import Path
nb=json.loads(Path('/tf/wjy/MMUL.ipynb').read_text())
source=next(''.join(c['source']) for c in nb['cells'] if ''.join(c.get('source',[])).lstrip().startswith('quant_code =') and 'def load_awq_model' in ''.join(c['source']))
exec(compile(source,'MMLU_AWQ','exec'))

RuntimeError: 评测失败，末尾诊断：  File "/tf/wjy/.venv-quant/lib/python3.11/site-packages/transformers/utils/generic.py", line 965, in wrapper
    output = func(self, *args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tf/wjy/.venv-quant/lib/python3.11/site-packages/transformers/models/llama/modeling_llama.py", line 571, in forward
    layer_outputs = decoder_layer(
                    ^^^^^^^^^^^^^^
  File "/tf/wjy/.venv-quant/lib/python3.11/site-packages/torch/nn/modules/module.py", line 1739, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tf/wjy/.venv-quant/lib/python3.11/site-packages/torch/nn/modules/module.py", line 1750, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tf/wjy/.venv-quant/lib/python3.11/site-packages/transformers/models/llama/modeling_llama.py", line 334, in forward
    hidden_states = self.mlp(hidden_states)
                    ^^

In [2]:
test = load_from_disk(ROOT / 'mmlu_fixed_1000')
demos = load_from_disk(ROOT / 'mmlu_fixed_5shot')
assert len(test) == 1000 and len(demos) == 285
assert not ({(x['subject'],x['question'].strip()) for x in test} & {(x['subject'],x['question'].strip()) for x in demos})
letters = 'ABCD'
by_subject = {}
for d in demos: by_subject.setdefault(d['subject'], []).append(d)
assert all(len(by_subject[r['subject']]) == 5 for r in test)
def question(r):
    return r['question'].strip() + '\n' + '\n'.join(f'{letters[i]}. {s}' for i,s in enumerate(r['choices']))
def messages(r):
    msg = [{'role':'system','content':f"Answer the multiple-choice question about {r['subject'].replace('_',' ')}. Reply with only the letter A, B, C, or D."}]
    for d in by_subject[r['subject']]:
        msg += [{'role':'user','content':question(d)}, {'role':'assistant','content':letters[int(d['answer'])]}]
    return msg + [{'role':'user','content':question(r)}]
label_ids = [tokenizer.encode(x, add_special_tokens=False) for x in letters]
assert all(len(x) == 1 for x in label_ids)
label_ids = [x[0] for x in label_ids]
correct = 0
processed = 0
with torch.inference_mode():
    for start in range(0, len(test), 4):
        rows = [test[i] for i in range(start, min(start+4,len(test)))]
        prompts = [tokenizer.apply_chat_template(messages(r),tokenize=False,add_generation_prompt=True) for r in rows]
        inputs = tokenizer(prompts,return_tensors='pt',padding=True,add_special_tokens=False)
        assert inputs.input_ids.shape[1] <= model.config.max_position_embeddings
        inputs = {k:v.to(model.device) for k,v in inputs.items()}
        logits = model(**inputs, use_cache=False).logits[:, -1, label_ids]
        predicted = logits.argmax(-1).tolist()
        correct += sum(p == int(r['answer']) for p,r in zip(predicted,rows))
        processed += len(rows)
accuracy = correct / len(test)
print(f'MMLU | FP16 | {correct}/{len(test)} | 准确率: {accuracy:.2%}')
release_model()


MMLU | FP16 | 658/1000 | 准确率: 65.80%


In [ ]:
quant_code = r'''
import os, sysconfig
from pathlib import Path
_nvroot = Path(sysconfig.get_paths()['purelib']) / 'nvidia'
os.environ['CPATH'] = os.pathsep.join((str(p) for p in _nvroot.glob('*/include')))
os.environ['LIBRARY_PATH'] = os.pathsep.join((str(p) for p in _nvroot.glob('*/lib')))
os.environ['MAX_JOBS'] = '2'
os.environ['TORCH_CUDA_ARCH_LIST'] = '8.6;8.9'
GPU = 0
import os
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import gc, re, ast, textwrap, warnings
from pathlib import Path
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from transformers.utils import logging as hf_logging
from optimum.quanto import quantize, freeze, qint8, quantization_map
from optimum.quanto.nn import QModuleMixin
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
warnings.filterwarnings('ignore', category=UserWarning)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
ROOT = Path('/tf/wjy')
MODEL_PATH = ROOT / 'models/Meta-Llama-3-8B-Instruct'
assert torch.cuda.is_available(), 'CUDA is required'
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True, padding_side='left')
tokenizer.pad_token = tokenizer.eos_token
EOS_IDS = list({tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids('<|eot_id|>')})

def release_model():
    global model
    if 'model' in globals():
        del model
    gc.collect()
    torch.cuda.empty_cache()

def load_quantized_model(bits):
    global model
    assert bits == 8
    release_model()
    set_seed(42)
    model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, local_files_only=True, torch_dtype=torch.float16, device_map={'': GPU}, attn_implementation='sdpa').eval()
    expected = [name for name, module in model.named_modules() if name.startswith('model.layers.') and isinstance(module, torch.nn.Linear)]
    assert len(expected) == model.config.num_hidden_layers * 7
    qtype = qint8
    quantize(model, weights=qtype, activations=None, include=expected)
    freeze(model)
    config = quantization_map(model)
    assert set(config) == set(expected)
    assert all((c == {'weights': f'qint{bits}', 'activations': 'none'} for c in config.values()))
    modules = [model.get_submodule(name) for name in expected]
    assert all((m.frozen and m.weight.dtype == torch.float16 for m in modules))
    assert model.model.embed_tokens.weight.dtype == model.lm_head.weight.dtype == torch.float16
    seen = set()

    def check_input(module, args):
        assert args[0].dtype == torch.float16, 'Activation is not FP16'
        seen.add(id(module))
    hooks = [m.register_forward_pre_hook(check_input) for m in modules]
    try:
        sample = tokenizer('Hello', return_tensors='pt').to(model.device)
        with torch.inference_mode():
            logits = model(**sample, use_cache=False).logits
        assert len(seen) == len(modules) and torch.isfinite(logits).all()
    finally:
        for hook in hooks:
            hook.remove()
    gc.collect()
    torch.cuda.empty_cache()

@torch.inference_mode()
def generate_batch(messages, max_new_tokens):
    prompts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages]
    inputs = tokenizer(prompts, return_tensors='pt', padding=True, add_special_tokens=False)
    assert inputs.input_ids.shape[1] + max_new_tokens <= model.config.max_position_embeddings
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    output = model.generate(**inputs, do_sample=False, temperature=None, top_p=None, max_new_tokens=max_new_tokens, eos_token_id=EOS_IDS, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.batch_decode(output[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def run_evaluation():
    test = load_from_disk(ROOT / 'mmlu_fixed_1000')
    demos = load_from_disk(ROOT / 'mmlu_fixed_5shot')
    assert len(test) == 1000 and len(demos) == 285
    assert not {(x['subject'], x['question'].strip()) for x in test} & {(x['subject'], x['question'].strip()) for x in demos}
    letters = 'ABCD'
    by_subject = {}
    for d in demos:
        by_subject.setdefault(d['subject'], []).append(d)
    assert all((len(by_subject[r['subject']]) == 5 for r in test))

    def question(r):
        return r['question'].strip() + '\n' + '\n'.join((f'{letters[i]}. {s}' for i, s in enumerate(r['choices'])))

    def messages(r):
        msg = [{'role': 'system', 'content': f"Answer the multiple-choice question about {r['subject'].replace('_', ' ')}. Reply with only the letter A, B, C, or D."}]
        for d in by_subject[r['subject']]:
            msg += [{'role': 'user', 'content': question(d)}, {'role': 'assistant', 'content': letters[int(d['answer'])]}]
        return msg + [{'role': 'user', 'content': question(r)}]
    label_ids = [tokenizer.encode(x, add_special_tokens=False) for x in letters]
    assert all((len(x) == 1 for x in label_ids))
    label_ids = [x[0] for x in label_ids]
    correct = 0
    processed = 0
    with torch.inference_mode():
        for start in range(0, len(test), 4):
            rows = [test[i] for i in range(start, min(start + 4, len(test)))]
            prompts = [tokenizer.apply_chat_template(messages(r), tokenize=False, add_generation_prompt=True) for r in rows]
            inputs = tokenizer(prompts, return_tensors='pt', padding=True, add_special_tokens=False)
            assert inputs.input_ids.shape[1] <= model.config.max_position_embeddings
            inputs = {k: v.to(model.device) for k, v in inputs.items()}
            logits = model(**inputs, use_cache=False).logits[:, -1, label_ids]
            predicted = logits.argmax(-1).tolist()
            correct += sum((p == int(r['answer']) for p, r in zip(predicted, rows)))
            processed += len(rows)
    return (correct, len(test))
try:
    load_quantized_model(8)
    correct, total = run_evaluation()
    print(f'MMLU | Quanto W8A16 | {correct}/{total} | accuracy: {correct / total:.2%}', flush=True)
finally:
    release_model()
'''
import os, subprocess
from pathlib import Path
from collections import deque
_root = Path('/tf/wjy')
_python = _root / '.venv-quant/bin/python'
_env = os.environ.copy()
_env['CUDA_VISIBLE_DEVICES'] = '1'
_env['PATH'] = str(_python.parent) + os.pathsep + _env.get('PATH', '')
_env['PYTHONWARNINGS'] = 'ignore'
for key, folder in [('TMPDIR', '.tmp'), ('HF_HOME', '.cache/huggingface'), ('TORCH_HOME', '.cache/torch'), ('TORCH_EXTENSIONS_DIR', '.cache/torch_extensions')]:
    directory = _root / folder
    directory.mkdir(parents=True, exist_ok=True)
    _env[key] = str(directory)
_tail = deque(maxlen=25)
with subprocess.Popen([str(_python), '-u', '-c', quant_code], cwd=str(_root), env=_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as _p:
    try:
        for line in _p.stdout:
            _tail.append(line)
            if line.startswith(('HumanEval |', 'MathQA |', 'MMLU |')):
                print(line, end='', flush=True)
        if _p.wait() != 0:
            raise RuntimeError('评测失败，末尾诊断：' + ''.join(_tail))
    except BaseException:
        if _p.poll() is None:
            _p.terminate()
            _p.wait()
        raise

In [1]:
quant_code = r'''
import os, sysconfig
from pathlib import Path
_nvroot = Path(sysconfig.get_paths()['purelib']) / 'nvidia'
os.environ['CPATH'] = os.pathsep.join((str(p) for p in _nvroot.glob('*/include')))
os.environ['LIBRARY_PATH'] = os.pathsep.join((str(p) for p in _nvroot.glob('*/lib')))
os.environ['MAX_JOBS'] = '2'
os.environ['TORCH_CUDA_ARCH_LIST'] = '8.6;8.9'
GPU = 0
import os
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import gc, re, ast, textwrap, warnings
from pathlib import Path
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
warnings.filterwarnings('ignore', category=UserWarning)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
ROOT = Path('/tf/wjy')
MODEL_PATH = ROOT / 'models/Meta-Llama-3-8B-Instruct'
assert torch.cuda.is_available(), 'CUDA is required'
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True, padding_side='left')
tokenizer.pad_token = tokenizer.eos_token
EOS_IDS = list({tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids('<|eot_id|>')})

def release_model():
    global model
    if 'model' in globals():
        del model
    gc.collect()
    torch.cuda.empty_cache()

@torch.inference_mode()
def generate_batch(messages, max_new_tokens):
    prompts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages]
    inputs = tokenizer(prompts, return_tensors='pt', padding=True, add_special_tokens=False)
    assert inputs.input_ids.shape[1] + max_new_tokens <= model.config.max_position_embeddings
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    output = model.generate(**inputs, do_sample=False, temperature=None, top_p=None, max_new_tokens=max_new_tokens, eos_token_id=EOS_IDS, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.batch_decode(output[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def run_evaluation():
    test = load_from_disk(ROOT / 'mmlu_fixed_1000')
    demos = load_from_disk(ROOT / 'mmlu_fixed_5shot')
    assert len(test) == 1000 and len(demos) == 285
    assert not {(x['subject'], x['question'].strip()) for x in test} & {(x['subject'], x['question'].strip()) for x in demos}
    letters = 'ABCD'
    by_subject = {}
    for d in demos:
        by_subject.setdefault(d['subject'], []).append(d)
    assert all((len(by_subject[r['subject']]) == 5 for r in test))

    def question(r):
        return r['question'].strip() + '\n' + '\n'.join((f'{letters[i]}. {s}' for i, s in enumerate(r['choices'])))

    def messages(r):
        msg = [{'role': 'system', 'content': f"Answer the multiple-choice question about {r['subject'].replace('_', ' ')}. Reply with only the letter A, B, C, or D."}]
        for d in by_subject[r['subject']]:
            msg += [{'role': 'user', 'content': question(d)}, {'role': 'assistant', 'content': letters[int(d['answer'])]}]
        return msg + [{'role': 'user', 'content': question(r)}]
    label_ids = [tokenizer.encode(x, add_special_tokens=False) for x in letters]
    assert all((len(x) == 1 for x in label_ids))
    label_ids = [x[0] for x in label_ids]
    correct = 0
    processed = 0
    with torch.inference_mode():
        for start in range(0, len(test), 4):
            rows = [test[i] for i in range(start, min(start + 4, len(test)))]
            prompts = [tokenizer.apply_chat_template(messages(r), tokenize=False, add_generation_prompt=True) for r in rows]
            inputs = tokenizer(prompts, return_tensors='pt', padding=True, add_special_tokens=False)
            assert inputs.input_ids.shape[1] <= model.config.max_position_embeddings
            inputs = {k: v.to(model.device) for k, v in inputs.items()}
            logits = model(**inputs, use_cache=False).logits[:, -1, label_ids]
            predicted = logits.argmax(-1).tolist()
            correct += sum((p == int(r['answer']) for p, r in zip(predicted, rows)))
            processed += len(rows)
    return (correct, len(test))

def load_awq_model():
    global model
    from datasets import load_dataset
    from awq import AutoAWQForCausalLM
    from awq.modules.linear import WQLinear_GEMM
    set_seed(42)
    raw = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='train', cache_dir=str(ROOT / '.cache/huggingface/datasets'))
    calibration = []
    seen_text = set()
    for row in raw:
        text = row['text'].strip()
        ids = tokenizer.encode(text, add_special_tokens=False)
        if 32 <= len(ids) <= 512 and text not in seen_text:
            calibration.append(text)
            seen_text.add(text)
        if len(calibration) == 128:
            break
    assert len(calibration) == 128
    awq_model = AutoAWQForCausalLM.from_pretrained(str(MODEL_PATH), torch_dtype=torch.float16, trust_remote_code=False, local_files_only=True, device_map={'': 'cpu'}, attn_implementation='sdpa')
    awq_model.quantize(tokenizer, quant_config={'zero_point': True, 'q_group_size': 128, 'w_bit': 4, 'version': 'GEMM'}, calib_data=calibration, max_calib_samples=128, max_calib_seq_len=512, n_parallel_calib_samples=4, max_chunk_memory=268435456, export_compatible=False)
    model = awq_model.model.to('cuda:0').eval()
    if hasattr(awq_model, 'quantizer'):
        del awq_model.quantizer
    layers = [m for m in model.modules() if isinstance(m, WQLinear_GEMM)]
    assert len(layers) == model.config.num_hidden_layers * 7
    assert all((m.w_bit == 4 and m.qweight.dtype == torch.int32 and (m.scales.dtype == torch.float16) for m in layers))
    seen = set()

    def verify(m, args):
        assert args[0].dtype == torch.float16
        seen.add(id(m))
    hooks = [m.register_forward_pre_hook(verify) for m in layers]
    try:
        with torch.inference_mode():
            logits = model(**tokenizer('Hello', return_tensors='pt').to('cuda:0'), use_cache=False).logits
        assert len(seen) == len(layers) and torch.isfinite(logits).all()
    finally:
        for hook in hooks:
            hook.remove()
    gc.collect()
    torch.cuda.empty_cache()
try:
    load_awq_model()
    correct, total = run_evaluation()
    print(f'MMLU | AWQ W4A16 | {correct}/{total} | accuracy: {correct / total:.2%}', flush=True)
finally:
    release_model()
'''
import os, subprocess
from pathlib import Path
from collections import deque
_root = Path('/tf/wjy')
_python = _root / '.venv-quant/bin/python'
_env = os.environ.copy()
_env['CUDA_VISIBLE_DEVICES'] = '1'
_env['PATH'] = str(_python.parent) + os.pathsep + _env.get('PATH', '')
_env['PYTHONWARNINGS'] = 'ignore'
for key, folder in [('TMPDIR', '.tmp'), ('HF_HOME', '.cache/huggingface'), ('TORCH_HOME', '.cache/torch'), ('TORCH_EXTENSIONS_DIR', '.cache/torch_extensions')]:
    directory = _root / folder
    directory.mkdir(parents=True, exist_ok=True)
    _env[key] = str(directory)
_tail = deque(maxlen=25)
with subprocess.Popen([str(_python), '-u', '-c', quant_code], cwd=str(_root), env=_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as _p:
    try:
        for line in _p.stdout:
            _tail.append(line)
            if line.startswith(('HumanEval |', 'MathQA |', 'MMLU |')):
                print(line, end='', flush=True)
        if _p.wait() != 0:
            raise RuntimeError('评测失败，末尾诊断：' + ''.join(_tail))
    except BaseException:
        if _p.poll() is None:
            _p.terminate()
            _p.wait()
        raise

MMLU | AWQ W4A16 | 648/1000 | accuracy: 64.80%


In [11]:
import ast
import json
from pathlib import Path

notebook_path = Path("/tf/wjy/MMUL.ipynb")
notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
candidates = []

for cell_index, cell in enumerate(notebook["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", "")
    source = "".join(source) if isinstance(source, list) else source

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in tree.body:
        if not isinstance(node, ast.Assign):
            continue
        if not any(
            isinstance(t, ast.Name) and t.id == "quant_code"
            for t in node.targets
        ):
            continue
        if not (
            isinstance(node.value, ast.Constant)
            and isinstance(node.value.value, str)
        ):
            continue

        value = node.value.value
        if (
            "mmlu_fixed_1000" in value
            and "def load_awq_model" in value
            and "awq_model.quantize(" in value
        ):
            candidates.append((cell_index + 1, value))

unique = {}
for cell_number, value in candidates:
    unique.setdefault(value, []).append(cell_number)

if len(unique) != 1:
    raise RuntimeError(
        f"找到 {len(unique)} 份不同的 MMLU AWQ 代码，"
        f"对应单元格：{[n for n, _ in candidates]}。"
        "请核对，暂不自动选择。"
    )

quant_code = next(iter(unique))
print("已提取 quant_code，来源单元格：", unique[quant_code])
print("仅提取字符串，没有执行量化或评测。")

已提取 quant_code，来源单元格： [5]
仅提取字符串，没有执行量化或评测。


In [8]:
import subprocess

for args in [
    ["nvidia-smi", "--query-gpu=index,memory.total,memory.used,memory.free",
     "--format=csv"],
    ["ps", "-p", "56793", "-o", "pid,ppid,args"],
    ["readlink", "/proc/56793/cwd"],
]:
    result = subprocess.run(args, capture_output=True, text=True)
    print(result.stdout or result.stderr)

index, memory.total [MiB], memory.used [MiB], memory.free [MiB]
0, 49140 MiB, 35973 MiB, 12537 MiB
1, 49140 MiB, 1 MiB, 48539 MiB

    PID    PPID COMMAND




In [14]:
import subprocess

# 从当前 notebook 的执行历史中提取刚才的只读检查代码
check_source = next(
    (
        source
        for source in reversed(In[:-1])
        if "from safetensors import safe_open" in source
        and "total_codes = sum(records.values())" in source
        and "expected_shape" in source
    ),
    None,
)

if check_source is None:
    raise RuntimeError("未找到检查代码。请先运行一次之前的检查单元格，再运行本单元格。")

result = subprocess.run(
    ["/tf/wjy/.venv-quant/bin/python", "-u", "-c", check_source],
    cwd="/tf/wjy",
    capture_output=True,
    text=True,
)

print(result.stdout, end="")
if result.stderr:
    print(result.stderr, end="")

if result.returncode != 0:
    raise RuntimeError(f"检查失败，退出码={result.returncode}")

量化配置： {'bits': 4, 'group_size': 128, 'modules_to_not_convert': None, 'quant_method': 'awq', 'version': 'gemm', 'zero_point': True}
量化矩阵数： 224
有效 INT4 code 数： 6979321856
有效 payload bit 数： 27917287424
padding nibble：0（已核对存储容量与逻辑权重数）
目标 1→0 翻转数： 83751862
实际目标 BER： 0.002999999990256933
检查通过；尚未注错。


In [15]:
import subprocess

code = r'''
import hashlib
import json
import re
import numpy as np
from pathlib import Path
from safetensors import safe_open

root = Path("/tf/wjy/models/Meta-Llama-3-8B-Instruct-AWQ-W4A16-g128")
pattern = re.compile(
    r"model\.layers\.\d+\."
    r"(?:self_attn\.(?:q_proj|k_proj|v_proj|o_proj)|"
    r"mlp\.(?:gate_proj|up_proj|down_proj))\.qweight"
)
popcount = np.array([i.bit_count() for i in range(256)], dtype=np.uint8)

records = []
for shard in sorted(root.glob("*.safetensors")):
    with safe_open(str(shard), framework="pt", device="cpu") as f:
        for name in sorted(f.keys()):
            if not pattern.fullmatch(name):
                continue

            tensor = f.get_tensor(name).contiguous()
            raw = tensor.numpy().view(np.uint8).reshape(-1)
            ones = 0
            digest = hashlib.sha256()

            for start in range(0, raw.size, 8 * 1024 * 1024):
                chunk = raw[start:start + 8 * 1024 * 1024]
                ones += int(popcount[chunk].sum(dtype=np.uint64))
                digest.update(chunk.tobytes())

            records.append({
                "name": name,
                "shape": list(tensor.shape),
                "total_bits": int(raw.size) * 8,
                "total_ones": ones,
                "qweight_sha256": digest.hexdigest(),
            })
            print(
                f"COUNT {len(records)}/224 | {name} | ones={ones}",
                flush=True,
            )
            del raw, tensor

records.sort(key=lambda r: r["name"])
assert len(records) == 224

total_bits = sum(r["total_bits"] for r in records)
total_ones = sum(r["total_ones"] for r in records)
target = (total_bits * 3 + 500) // 1000

assert total_bits == 27917287424
assert total_ones >= target

metadata_hash = hashlib.sha256(
    json.dumps(records, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()

print("\nTOTAL_BITS:", total_bits, flush=True)
print("TOTAL_ONES:", total_ones, flush=True)
print("ONES_RATIO:", total_ones / total_bits, flush=True)
print("TARGET_CHANGED_BITS:", target, flush=True)
print("METADATA_SHA256:", metadata_hash, flush=True)
print("候选数量检查通过；未修改权重，尚未生成 mask。", flush=True)
'''

process = subprocess.Popen(
    ["/tf/wjy/.venv-quant/bin/python", "-u", "-c", code],
    cwd="/tf/wjy",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in process.stdout:
        print(line, end="", flush=True)
    rc = process.wait()
    if rc:
        raise RuntimeError(f"统计失败，退出码={rc}")
except BaseException:
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    raise
finally:
    process.stdout.close()

COUNT 1/224 | model.layers.0.mlp.down_proj.qweight | ones=117439223
COUNT 2/224 | model.layers.0.mlp.gate_proj.qweight | ones=118008399
COUNT 3/224 | model.layers.0.mlp.up_proj.qweight | ones=118094516
COUNT 4/224 | model.layers.0.self_attn.k_proj.qweight | ones=8388312
COUNT 5/224 | model.layers.0.self_attn.o_proj.qweight | ones=33564054
COUNT 6/224 | model.layers.0.self_attn.q_proj.qweight | ones=33532377
COUNT 7/224 | model.layers.0.self_attn.v_proj.qweight | ones=8347184
COUNT 8/224 | model.layers.1.mlp.down_proj.qweight | ones=117454014
COUNT 9/224 | model.layers.1.mlp.gate_proj.qweight | ones=118050065
COUNT 10/224 | model.layers.1.mlp.up_proj.qweight | ones=118075095
COUNT 11/224 | model.layers.1.self_attn.k_proj.qweight | ones=8381585
COUNT 12/224 | model.layers.1.self_attn.o_proj.qweight | ones=33649140
COUNT 13/224 | model.layers.1.self_attn.q_proj.qweight | ones=33544419
COUNT 14/224 | model.layers.1.self_attn.v_proj.qweight | ones=8420845
COUNT 15/224 | model.layers.10.mlp.

In [17]:
import os
import subprocess

mask_program_v2 = r'''
import gc
import hashlib
import json
import shutil
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

ROOT = Path("/tf/wjy")
MODEL = ROOT / "models/Meta-Llama-3-8B-Instruct-AWQ-W4A16-g128"
SEEDS = (2026091201, 2026091202, 2026091203,
         2026091204, 2026091205)

# 使用此前已经核对的 clean 模型统计值。
TOTAL_BITS = 27917287424
TOTAL_ONES = 14019732397
TARGET = (TOTAL_BITS * 3 + 500) // 1000
METADATA_SHA256 = (
    "f21c4950038d762dbc404987fcb455315e1b2473ed893f25da1c8ef43fe2a60b"
)

assert MODEL.is_dir(), MODEL
assert TARGET == 83751862
assert 0 < TARGET <= TOTAL_ONES

required = len(SEEDS) * TARGET * 8 + 1024**3
assert shutil.disk_usage(ROOT).free >= required, "磁盘空间不足"

OUT = Path(tempfile.mkdtemp(
    prefix="awq_int4_1to0_ber003_masks_v2_",
    dir=str(ROOT),
))
print(f"MASK_DIR: {OUT}", flush=True)
print(
    f"TOTAL_BITS={TOTAL_BITS} | TOTAL_ONES={TOTAL_ONES} "
    f"| TARGET={TARGET} | BER={TARGET / TOTAL_BITS:.15f}",
    flush=True,
)

def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for data in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(data)
    return h.hexdigest()

def uniform_subset(n, k, seed):
    # 与 INT8 脚本相同的抽样方法。
    # 所有整数都是“初始为 1 的 bit”的编号。
    rng = np.random.Generator(np.random.PCG64(seed))
    selected = np.empty(0, dtype=np.int64)
    attempt = 0

    while selected.size < k:
        attempt += 1
        need = k - selected.size
        print(
            f"DRAW seed={seed} | pass={attempt} | drawing={need}",
            flush=True,
        )
        proposed = rng.integers(0, n, size=need, dtype=np.int64)

        if selected.size == 0:
            selected = np.unique(proposed)
        else:
            selected = np.union1d(selected, proposed)

        del proposed
        print(
            f"SELECTED seed={seed} | unique={selected.size}/{k}",
            flush=True,
        )

    return selected

# 必须在应用 mask 时保持同样的候选顺序：
# module 名字典序 -> contiguous qweight C 顺序 ->
# 每个 int32 的 bit 0..31 -> 仅初始为 1 的 bit，从 0 编号。
# 本阶段使用已核对的总数和元数据哈希；应用前须重新核对 clean 模型。

for round_id, seed in enumerate(SEEDS, 1):
    folder = OUT / f"seed_{seed}"
    folder.mkdir()

    print(f"START round={round_id}/5 seed={seed}", flush=True)
    ranks = uniform_subset(TOTAL_ONES, TARGET, seed)

    assert ranks.dtype == np.int64
    assert ranks.size == TARGET

    previous = -1
    block = 1_000_000
    for start in range(0, TARGET, block):
        part = ranks[start:start + block]
        assert int(part[0]) > previous
        assert int(part[-1]) < TOTAL_ONES
        assert np.all(part[1:] > part[:-1])
        previous = int(part[-1])
    del part

    temporary = folder / "selected_one_bit_ranks.partial.npy"
    final = folder / "selected_one_bit_ranks.npy"

    print(f"SAVE seed={seed} | writing mask", flush=True)
    with temporary.open("xb") as f:
        np.save(f, ranks, allow_pickle=False)
        f.flush()
        os_fileno = f.fileno()
        # 确保文件写入后再进行发布。
        import os
        os.fsync(os_fileno)

    del ranks
    gc.collect()

    # 从磁盘重新读取，检查实际保存的内容。
    saved = np.load(temporary, mmap_mode="r", allow_pickle=False)
    assert saved.dtype == np.int64 and saved.shape == (TARGET,)
    previous = -1
    for start in range(0, TARGET, block):
        part = saved[start:start + block]
        assert int(part[0]) > previous
        assert int(part[-1]) < TOTAL_ONES
        assert np.all(part[1:] > part[:-1])
        previous = int(part[-1])
    del part, saved

    temporary.rename(final)
    mask_sha = sha256_file(final)

    manifest = {
        "seed": seed,
        "status": "mask_ready",
        "model_path": str(MODEL),
        "direction": "1to0",
        "quantization": {
            "quant_method": "awq",
            "bits": 4,
            "group_size": 128,
            "zero_point": True,
            "version": "gemm",
        },
        "matrix_count": 224,
        "total_valid_payload_bits": TOTAL_BITS,
        "total_ones": TOTAL_ONES,
        "target_changed_bits": TARGET,
        "actual_changed_bits": None,
        "actual_ber": None,
        "target_actual_ber": TARGET / TOTAL_BITS,
        "clean_metadata_sha256": METADATA_SHA256,
        "clean_metadata_verified_this_run": False,
        "mask_file": final.name,
        "mask_sha256": mask_sha,
        "rank_order": (
            "module_name_lexicographic; contiguous_qweight_C_order;"
            " int32_bit_0_to_31; initial_ones_only; zero_based"
        ),
        "sampling": "global_uniform_integer_ranks_deduplicate_until_k",
        "rng": "numpy.PCG64",
        "numpy_version": np.__version__,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }
    with (folder / "mask_manifest.json").open("x", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

    print(
        f"READY seed={seed} | count={TARGET} | SHA256={mask_sha}",
        flush=True,
    )

print(f"ALL_MASKS_READY: {OUT}", flush=True)
print("五份 mask 已生成并校验；尚未修改模型或运行评测。", flush=True)
'''

process = subprocess.Popen(
    ["/tf/wjy/.venv-quant/bin/python", "-u", "-c", mask_program_v2],
    cwd="/tf/wjy",
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    bufsize=1,
)

print(f"mask 生成进程 PID={process.pid}", flush=True)
MASK_DIR = None

try:
    for line in process.stdout:
        print(line, end="", flush=True)
        if line.startswith("MASK_DIR: "):
            MASK_DIR = line.split(": ", 1)[1].strip()

    rc = process.wait()
    if rc != 0:
        raise RuntimeError(f"mask 生成失败，退出码={rc}")
except BaseException:
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    raise
finally:
    process.stdout.close()

print("后续使用的 mask 目录：", MASK_DIR)

mask 生成进程 PID=133922
MASK_DIR: /tf/wjy/awq_int4_1to0_ber003_masks_v2_t5n573zd
TOTAL_BITS=27917287424 | TOTAL_ONES=14019732397 | TARGET=83751862 | BER=0.002999999990257
START round=1/5 seed=2026091201
DRAW seed=2026091201 | pass=1 | drawing=83751862
SELECTED seed=2026091201 | unique=83501927/83751862
DRAW seed=2026091201 | pass=2 | drawing=249935
SELECTED seed=2026091201 | unique=83750360/83751862
DRAW seed=2026091201 | pass=3 | drawing=1502
SELECTED seed=2026091201 | unique=83751851/83751862
DRAW seed=2026091201 | pass=4 | drawing=11
SELECTED seed=2026091201 | unique=83751862/83751862
SAVE seed=2026091201 | writing mask
READY seed=2026091201 | count=83751862 | SHA256=846a26e1ac504599d72483792d0de32116c3536a45c1d422ad6bb31f2cb68141
START round=2/5 seed=2026091202
DRAW seed=2026091202 | pass=1 | drawing=83751862
SELECTED seed=2026091202 | unique=83501224/83751862
DRAW seed=2026091202 | pass=2 | drawing=250638
SELECTED seed=2026091202 | unique=83750327/83751862
DRAW seed=2026091202 | pass

In [18]:
import subprocess

check_code = r'''
import hashlib
import json
from pathlib import Path
import numpy as np

root = Path("/tf/wjy/awq_int4_1to0_ber003_masks_v2_t5n573zd")
target = 83751862
total_ones = 14019732397
metadata_sha = "f21c4950038d762dbc404987fcb455315e1b2473ed893f25da1c8ef43fe2a60b"

for seed in range(2026091201, 2026091206):
    folder = root / f"seed_{seed}"
    manifest = json.loads(
        (folder / "mask_manifest.json").read_text(encoding="utf-8")
    )
    path = folder / "selected_one_bit_ranks.npy"

    assert manifest["seed"] == seed
    assert manifest["status"] == "mask_ready"
    assert manifest["target_changed_bits"] == target
    assert manifest["total_valid_payload_bits"] == 27917287424
    assert manifest["total_ones"] == total_ones
    assert manifest["clean_metadata_sha256"] == metadata_sha

    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    assert h.hexdigest() == manifest["mask_sha256"], f"哈希不匹配：{seed}"

    ranks = np.load(path, mmap_mode="r", allow_pickle=False)
    assert ranks.dtype == np.int64
    assert ranks.shape == (target,)
    previous = -1
    for start in range(0, target, 1_000_000):
        part = ranks[start:start + 1_000_000]
        assert int(part[0]) > previous
        assert int(part[-1]) < total_ones
        assert np.all(part[1:] > part[:-1])
        previous = int(part[-1])
    del part, ranks

    print(f"PASS seed={seed} | count={target} | SHA256={h.hexdigest()}",
          flush=True)

print("ALL_5_MASKS_VERIFIED", flush=True)
'''

subprocess.run(
    ["/tf/wjy/.venv-quant/bin/python", "-u", "-c", check_code],
    cwd="/tf/wjy",
    check=True,
)

PASS seed=2026091201 | count=83751862 | SHA256=846a26e1ac504599d72483792d0de32116c3536a45c1d422ad6bb31f2cb68141
PASS seed=2026091202 | count=83751862 | SHA256=01db7364d8ca27e0a5cbe1c214c73ba366f9082b890f77f6332fe2891c2e368a
PASS seed=2026091203 | count=83751862 | SHA256=d753aaeeb461afb1d8522ecde1e998dd1a7b04a257a83d36ceafd8ddd5eeea5a
PASS seed=2026091204 | count=83751862 | SHA256=be2fe6157f256f1004042d64e3ae0f05c821f27eae5f9ffac154fd643ce6a72b
PASS seed=2026091205 | count=83751862 | SHA256=47ebc41b6ff8ce5ed9ea83615385a4c6d4c741d880d8caf2e21da40ba846b8da
ALL_5_MASKS_VERIFIED


CompletedProcess(args=['/tf/wjy/.venv-quant/bin/python', '-u', '-c', '\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\n\nroot = Path("/tf/wjy/awq_int4_1to0_ber003_masks_v2_t5n573zd")\ntarget = 83751862\ntotal_ones = 14019732397\nmetadata_sha = "f21c4950038d762dbc404987fcb455315e1b2473ed893f25da1c8ef43fe2a60b"\n\nfor seed in range(2026091201, 2026091206):\n    folder = root / f"seed_{seed}"\n    manifest = json.loads(\n        (folder / "mask_manifest.json").read_text(encoding="utf-8")\n    )\n    path = folder / "selected_one_bit_ranks.npy"\n\n    assert manifest["seed"] == seed\n    assert manifest["status"] == "mask_ready"\n    assert manifest["target_changed_bits"] == target\n    assert manifest["total_valid_payload_bits"] == 27917287424\n    assert manifest["total_ones"] == total_ones\n    assert manifest["clean_metadata_sha256"] == metadata_sha\n\n    h = hashlib.sha256()\n    with path.open("rb") as f:\n        for chunk in iter(lambda: f.read(8 * 1024

In [20]:
import ast
import os
import json
import hashlib
import subprocess
import tempfile
import statistics
import fcntl
from pathlib import Path

ROOT = Path("/tf/wjy")
PYTHON = ROOT / ".venv-quant/bin/python"
GPU = "1"  # 物理 GPU
MASK_ROOT = ROOT / "awq_int4_1to0_ber003_masks_v2_t5n573zd"
SEEDS = list(range(2026091201, 2026091206))

assert PYTHON.is_file()
assert MASK_ROOT.is_dir()

# 从已保存的 notebook 中提取原始 AWQ MMLU 评测函数。
# 不执行原来的量化代码，也不修改原单元格。
nb = json.loads((ROOT / "MMUL.ipynb").read_text(encoding="utf-8"))
candidates = {}

for cell in nb["cells"]:
    if cell.get("cell_type") != "code":
        continue
    try:
        outer = ast.parse("".join(cell.get("source", [])))
    except SyntaxError:
        continue

    for node in outer.body:
        if not isinstance(node, ast.Assign):
            continue
        if not any(
            isinstance(t, ast.Name) and t.id == "quant_code"
            for t in node.targets
        ):
            continue
        if not (
            isinstance(node.value, ast.Constant)
            and isinstance(node.value.value, str)
        ):
            continue

        text = node.value.value
        if "mmlu_fixed_1000" not in text:
            continue
        inner = ast.parse(text)
        funcs = {
            n.name: n for n in inner.body
            if isinstance(n, ast.FunctionDef)
        }
        if "load_awq_model" not in funcs or "run_evaluation" not in funcs:
            continue

        fn = funcs["run_evaluation"]
        canonical = ast.dump(fn, include_attributes=False)
        candidates[canonical] = fn

assert len(candidates) == 1, (
    f"找到 {len(candidates)} 种 AWQ MMLU 评测函数。"
    "请先保存包含 clean AWQ 代码的 notebook；存在不同版本时不要自动选择。"
)

eval_fn = next(iter(candidates.values()))
original_eval_source = ast.unparse(eval_fn)
EVAL_SHA = hashlib.sha256(original_eval_source.encode()).hexdigest()

# 只添加进度打印，不改评测计算。
has_progress = any(
    isinstance(n, ast.Call)
    and isinstance(n.func, ast.Name)
    and n.func.id == "print"
    for n in ast.walk(eval_fn)
)

if not has_progress:
    class AddProgress(ast.NodeTransformer):
        def __init__(self):
            self.hits = 0

        def visit_AugAssign(self, node):
            if isinstance(node.target, ast.Name) and node.target.id == "processed":
                self.hits += 1
                extra = ast.parse(
                    'print(f"PROGRESS {processed}/{len(test)} '
                    'correct={correct}", flush=True)'
                ).body[0]
                return [node, extra]
            return node

    patch = AddProgress()
    eval_fn = patch.visit(eval_fn)
    assert patch.hits == 1, "进度插入点不唯一，已停止"
    ast.fix_missing_locations(eval_fn)

eval_source = ast.unparse(eval_fn)

OUT = Path(tempfile.mkdtemp(prefix="mmlu_awq_int4_1to0_", dir=str(ROOT)))
(OUT / "original_evaluator.py").write_text(original_eval_source, encoding="utf-8")

worker_prefix = r'''
import os, sys, sysconfig, json, hashlib, gc, re, traceback
from pathlib import Path
from datetime import datetime, timezone

nvroot = Path(sysconfig.get_paths()["purelib"]) / "nvidia"
os.environ["CPATH"] = os.pathsep.join(str(p) for p in nvroot.glob("*/include"))
os.environ["LIBRARY_PATH"] = os.pathsep.join(str(p) for p in nvroot.glob("*/lib"))
os.environ["MAX_JOBS"] = "2"
os.environ["TORCH_CUDA_ARCH_LIST"] = "8.6;8.9"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, set_seed
from awq import AutoAWQForCausalLM
from awq.modules.linear import WQLinear_GEMM

ROOT = Path("/tf/wjy")
MODEL_PATH = ROOT / "models/Meta-Llama-3-8B-Instruct-AWQ-W4A16-g128"
TOKENIZER_PATH = ROOT / "models/Meta-Llama-3-8B-Instruct"
seed = int(sys.argv[1])
MASK_ROOT = Path(sys.argv[2])
OUT = Path(sys.argv[3])
EVAL_SHA = sys.argv[4]

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
set_seed(42)

def sha_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def raw_bytes(t):
    return t.detach().cpu().contiguous().view(torch.uint8).numpy().reshape(-1)

def tensor_hash(t):
    return hashlib.sha256(raw_bytes(t)).hexdigest()

def save_json(path, obj):
    tmp = path.with_suffix(".partial.json")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
    tmp.replace(path)

def protected_hashes(model, qkeys):
    # 包括 scales、qzeros、bias、嵌入、lm_head 及其余参数/缓冲区。
    tensors = dict(model.named_parameters())
    tensors.update(dict(model.named_buffers()))
    return {
        name: (str(t.dtype), list(t.shape), tensor_hash(t))
        for name, t in sorted(tensors.items())
        if name not in qkeys
    }

def inject(model, manifest, ranks):
    pop = np.array([i.bit_count() for i in range(256)], dtype=np.uint8)
    nth = np.zeros((256, 8), dtype=np.uint8)
    for value in range(256):
        bits = [b for b in range(8) if value & (1 << b)]
        nth[value, :len(bits)] = bits

    assert sys.byteorder == "little", "物理 bit 顺序要求 little-endian"
    layers = sorted(
        (name, m) for name, m in model.named_modules()
        if isinstance(m, WQLinear_GEMM)
    )
    expected = {
        f"model.layers.{i}.{suffix}"
        for i in range(32)
        for suffix in (
            "self_attn.q_proj", "self_attn.k_proj",
            "self_attn.v_proj", "self_attn.o_proj",
            "mlp.gate_proj", "mlp.up_proj", "mlp.down_proj",
        )
    }
    assert {name for name, _ in layers} == expected
    qkeys = {name + ".qweight" for name, _ in layers}
    protected_before = protected_hashes(model, qkeys)

    records = []
    total_bits = 0
    total_ones = 0
    for name, m in layers:
        assert m.w_bit == 4 and m.group_size == 128
        assert m.qweight.dtype == torch.int32
        assert m.scales.dtype == torch.float16
        assert m.qweight.numel() * 8 == m.in_features * m.out_features, (
            "发现 padding 或不同打包布局", name
        )
        raw = raw_bytes(m.qweight)
        ones = int(pop[raw].sum(dtype=np.int64))
        records.append({
            "name": name + ".qweight",
            "shape": list(m.qweight.shape),
            "total_bits": int(raw.size * 8),
            "total_ones": ones,
            "qweight_sha256": hashlib.sha256(raw).hexdigest(),
        })
        total_bits += int(raw.size * 8)
        total_ones += ones
        del raw

    metadata_sha = hashlib.sha256(
        json.dumps(records, sort_keys=True, separators=(",", ":")).encode()
    ).hexdigest()
    assert metadata_sha == manifest["clean_metadata_sha256"], (
        "clean 权重/布局与 mask 元数据不一致",
        metadata_sha,
        manifest["clean_metadata_sha256"],
    )
    assert total_bits == 27917287424
    assert total_ones == 14019732397
    target = (total_bits * 3 + 500) // 1000
    assert target == manifest["target_changed_bits"] == len(ranks)
    print(f"CLEAN AUDIT OK | matrices=224 | bits={total_bits} "
          f"| ones={total_ones}", flush=True)

    offset = 0
    verified = 0
    per_matrix = []
    with torch.no_grad():
        for name, m in layers:
            before = raw_bytes(m.qweight).copy()
            after = before.copy()
            matrix_count = 0

            for start in range(0, before.size, 1048576):
                chunk = before[start:start + 1048576]
                count = int(pop[chunk].sum(dtype=np.int64))
                lo, hi = np.searchsorted(ranks, [offset, offset + count])
                lo, hi = int(lo), int(hi)

                if hi > lo:
                    local = ranks[lo:hi] - offset
                    counts = pop[chunk]
                    cumulative = np.cumsum(counts, dtype=np.int64)
                    byte_ids = np.searchsorted(cumulative, local, side="right")
                    within = local - (cumulative[byte_ids] - counts[byte_ids])
                    bit_ids = nth[chunk[byte_ids], within]
                    clear_mask = np.zeros(chunk.size, dtype=np.uint8)
                    np.bitwise_or.at(
                        clear_mask, byte_ids,
                        np.left_shift(np.uint8(1), bit_ids),
                    )
                    after[start:start + chunk.size] = (
                        chunk & np.bitwise_not(clear_mask)
                    )
                offset += count
                matrix_count += hi - lo

            packed = torch.from_numpy(
                after.view(np.int32).reshape(tuple(m.qweight.shape))
            )
            m.qweight.copy_(packed.to(m.qweight.device))
            actual = raw_bytes(m.qweight)
            assert np.array_equal(actual, after), f"写回不一致：{name}"
            assert not np.any(np.bitwise_not(before) & actual), f"发现 0→1：{name}"
            changed = int(pop[before ^ actual].sum(dtype=np.int64))
            assert changed == matrix_count
            verified += changed
            per_matrix.append({"name": name, "actual_changed_bits": changed})
            print(f"INJECT {name} | changed={changed} "
                  f"| verified={verified}/{target}", flush=True)
            del before, after, actual, packed

    assert offset == total_ones and verified == target
    assert protected_before == protected_hashes(model, qkeys), (
        "qweight 以外的参数或缓冲区发生变化"
    )
    by_layer = {}
    for item in per_matrix:
        layer = ".".join(item["name"].split(".")[:3])
        by_layer[layer] = by_layer.get(layer, 0) + item["actual_changed_bits"]

    gc.collect()
    torch.cuda.empty_cache()
    return {
        "actual_changed_bits": verified,
        "total_bits": total_bits,
        "total_ones": total_ones,
        "actual_ber": verified / total_bits,
        "clean_metadata_sha256": metadata_sha,
        "per_matrix": per_matrix,
        "per_layer": by_layer,
        "protected_tensors_unchanged": True,
    }
'''

worker_suffix = r'''
result = {
    "task": "MMLU",
    "seed": seed,
    "status": "running",
    "correct": None,
    "total": 1000,
    "accuracy": None,
    "torch_dtype": "torch.float16",
    "evaluator_sha256": EVAL_SHA,
    "script_sha256": sha_file(Path(__file__)),
    "timestamp": datetime.now(timezone.utc).isoformat(),
}
result_path = OUT / f"seed_{seed}.json"
seen_inputs = set()
nonfinite = {"detected": False}
hooks = []

try:
    print(f"START seed={seed} PID={os.getpid()}", flush=True)
    folder = MASK_ROOT / f"seed_{seed}"
    manifest = json.loads((folder / "mask_manifest.json").read_text())
    mask_path = folder / "selected_one_bit_ranks.npy"
    assert manifest["seed"] == seed and manifest["status"] == "mask_ready"
    assert manifest["direction"] == "1to0"
    assert manifest["rank_order"] == (
        "module_name_lexicographic; contiguous_qweight_C_order;"
        " int32_bit_0_to_31; initial_ones_only; zero_based"
    )
    assert sha_file(mask_path) == manifest["mask_sha256"]
    result["mask_sha256"] = manifest["mask_sha256"]
    ranks = np.load(mask_path, mmap_mode="r", allow_pickle=False)
    assert ranks.dtype == np.int64 and ranks.shape == (83751862,)
    previous = -1
    for start in range(0, len(ranks), 1000000):
        part = ranks[start:start + 1000000]
        assert int(part[0]) > previous
        assert int(part[-1]) < 14019732397
        assert np.all(part[1:] > part[:-1])
        previous = int(part[-1])
    del part

    config = json.loads((MODEL_PATH / "config.json").read_text())
    qc = config["quantization_config"]
    assert qc["quant_method"].lower() == "awq"
    assert qc["bits"] == 4 and qc["group_size"] == 128
    assert qc["zero_point"] is True and qc["version"].lower() == "gemm"
    result["quantization_config"] = qc

    tokenizer = AutoTokenizer.from_pretrained(
        TOKENIZER_PATH, local_files_only=True, padding_side="left"
    )
    tokenizer.pad_token = tokenizer.eos_token

    print("LOAD clean saved AWQ W4A16", flush=True)
    wrapper = AutoAWQForCausalLM.from_quantized(
        str(MODEL_PATH),
        fuse_layers=False,
        device_map={"": "cuda:0"},
        torch_dtype=torch.float16,
        trust_remote_code=False,
        local_files_only=True,
        attn_implementation="sdpa",
    )
    model = wrapper.model.eval()
    result.update(inject(model, manifest, ranks))
    del ranks

    def check_input(module, args):
        assert args[0].dtype == torch.float16
        seen_inputs.add(id(module))

    def check_logits(module, args, output):
        if not torch.isfinite(output.logits[:, -1, :]).all().item():
            nonfinite["detected"] = True
            raise RuntimeError("Nonfinite logits：本轮无效，不计为 0 分")

    layers = [m for m in model.modules() if isinstance(m, WQLinear_GEMM)]
    hooks = [m.register_forward_pre_hook(check_input) for m in layers]
    hooks.append(model.register_forward_hook(check_logits))
    print("EVAL MMLU | 1000 questions | original 5-shot | batch=4", flush=True)

    correct, total = run_evaluation()
    assert total == 1000 and len(seen_inputs) == 224
    result.update(
        status="completed",
        correct=int(correct),
        total=int(total),
        accuracy=correct / total,
        nonfinite_logits=False,
    )
    print(f"RESULT seed={seed} | {correct}/{total} "
          f"| accuracy={correct / total:.2%}", flush=True)

except BaseException as exc:
    result.update(
        status="failed",
        error=f"{type(exc).__name__}: {exc}",
        nonfinite_logits=nonfinite["detected"],
    )
    raise

finally:
    for h in hooks:
        h.remove()
    result["finished_at"] = datetime.now(timezone.utc).isoformat()
    save_json(result_path, result)
'''

worker = worker_prefix + "\n" + eval_source + "\n" + worker_suffix
compile(worker, "<MMLU_AWQ_INT4>", "exec")
SCRIPT = OUT / "mmlu_worker.py"
SCRIPT.write_text(worker, encoding="utf-8")

env = os.environ.copy()
env.update(
    CUDA_VISIBLE_DEVICES=GPU,
    PYTHONWARNINGS="ignore",
    USE_TF="0",
    USE_FLAX="0",
    TOKENIZERS_PARALLELISM="false",
)
env["PATH"] = str(PYTHON.parent) + os.pathsep + env.get("PATH", "")
for key, folder in [
    ("TMPDIR", ".tmp"),
    ("HF_HOME", ".cache/huggingface"),
    ("TORCH_HOME", ".cache/torch"),
    ("TORCH_EXTENSIONS_DIR", ".cache/torch_extensions"),
]:
    directory = ROOT / folder
    directory.mkdir(parents=True, exist_ok=True)
    env[key] = str(directory)

print("结果目录：", OUT)
print("原始 MMLU 评测函数 SHA256：", EVAL_SHA)
print("物理 GPU：", GPU)

# 先显示当前资源情况；不终止任何已有进程。
subprocess.run(["nvidia-smi"], check=True)

results = []
lock_path = ROOT / ".tmp" / f"awq_int4_gpu{GPU}.lock"

with lock_path.open("a") as lock:
    try:
        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
    except BlockingIOError:
        raise RuntimeError("这个 GPU 已有使用同一锁的 AWQ 实验正在运行")

    for trial, seed in enumerate(SEEDS, 1):
        log_path = OUT / f"seed_{seed}.log"
        print(f"\nROUND {trial}/5 | seed={seed} | log={log_path}", flush=True)

        with log_path.open("x", encoding="utf-8") as log:
            process = subprocess.Popen(
                [str(PYTHON), "-u", str(SCRIPT), str(seed),
                 str(MASK_ROOT), str(OUT), EVAL_SHA],
                cwd=str(ROOT),
                env=env,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                encoding="utf-8",
                bufsize=1,
            )
            print("子进程 PID：", process.pid, flush=True)
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    log.flush()
                rc = process.wait()
            except BaseException:
                if process.poll() is None:
                    process.terminate()
                    try:
                        process.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        process.kill()
                        process.wait()
                raise
            finally:
                process.stdout.close()

        if rc != 0:
            raise RuntimeError(
                f"第 {trial} 轮失败，退出码={rc}；已停止后续轮次。日志：{log_path}"
            )

        result = json.loads((OUT / f"seed_{seed}.json").read_text())
        assert result["status"] == "completed"
        results.append(result)

scores = [r["accuracy"] for r in results]
summary = {
    "task": "MMLU",
    "results": [
        {k: r[k] for k in ("seed", "correct", "total", "accuracy")}
        for r in results
    ],
    "mean_accuracy": statistics.mean(scores),
    "population_std": statistics.pstdev(scores),
    "sample_std": statistics.stdev(scores),
}
(OUT / "summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)

print("\n五轮 MMLU 完成")
for r in results:
    print(f"seed={r['seed']} | {r['correct']}/{r['total']} "
          f"| {r['accuracy']:.2%}")
print(f"平均准确率：{summary['mean_accuracy']:.2%}")
print(f"总体标准差：{summary['population_std'] * 100:.4f} 个百分点")
print(f"样本标准差：{summary['sample_std'] * 100:.4f} 个百分点")
print("结果目录：", OUT)

结果目录： /tf/wjy/mmlu_awq_int4_1to0_i8o1w4fl
原始 MMLU 评测函数 SHA256： 7bd17f29695d1338b528baf006edfda06c5bc526b4256ef8442c2852030358b0
物理 GPU： 1
Sat Sep 19 13:29:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX 6000 Ada Gene...    On  |   00000000:AF:00.0 Off |                  Off |
| 88%   85C    P2            299W /  300W |   35973MiB /  49140MiB |     99%      Default |
| 